# CPU experiment-to-video integration
Run all cells on CPU. This notebook tests software integration, **not scientific video quality**. Dummy output holds test cards static. No model weights, GPU, API key, or cloud account are used.

Install from the video checkout with `%pip install -e ".[rag,notebook]"` (or point this command to its absolute path), then choose that Python kernel. Clone the dataset repository alongside it, using the branch that contains `scripts/plan_experiment.py`.

The dataset review gate reads existing review records; this notebook does not approve or change scientific evidence. Test cards are never represented as photographs of the experiment.

In [ ]:
from pathlib import Path
import os, json
from PIL import Image, ImageDraw
from IPython.display import Video, display
from edu_video import Catalog, VideoPipeline

# Override EDU_DATASET_DIR for any other checkout location.
cwd = Path.cwd()
video_root = cwd.parent if cwd.name == "notebooks" else cwd
DATASET_DIR = Path(os.environ.get("EDU_DATASET_DIR", str(video_root.parent / "chemistry-dataset"))).resolve()
RUN_DIR = Path(os.environ.get("EDU_RUN_DIR", str(video_root / "runs/notebook_demo"))).resolve()
catalog = Catalog(DATASET_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(DATASET_DIR)

## Search and explicit selection
Search returns candidates only. Change the selected ID after inspecting the results.

In [ ]:
decision = catalog.search("boiling point organic compound")
print(decision["status"])
for candidate in decision["candidates"]:
    print(candidate["id"], candidate["title"], round(candidate["score"], 3))
assert decision["clips"] == []

In [ ]:
EXPERIMENT_ID = "chem_12_101"  # Explicit selection; no automatic fallback.
plan = catalog.plan(EXPERIMENT_ID)
if plan["status"] != "ready":
    raise RuntimeError(json.dumps(plan, indent=2))
print("Evidence gate:", plan["review_status"])
print("Visual review:", plan["visual_review_status"])
for clip in plan["clips"]:
    print(clip["clip_id"], clip["scene_id"], clip["duration_seconds"], clip["source_passage_ids"])
    print(clip["motion_prompt"])
print("Sources:", json.dumps(plan["source_references"], indent=2))
plan_file = RUN_DIR / "selected_plan.json"
plan_file.write_text(json.dumps(plan, indent=2), encoding="utf-8")

## Assign scene images
These labeled CPU test cards are integration fixtures. Later replace each path with a reviewed laboratory photograph, stock image, or generated setup image and record its provenance. Continuing clips use the previous clip?s last frame.

In [ ]:
scene_images = {}
for clip in plan["clips"]:
    if not clip["new_scene"]:
        continue
    scene_id = clip["scene_id"]
    image_path = RUN_DIR / f"{scene_id}_test_card.png"
    card = Image.new("RGB", (320, 240), "#182b40")
    ImageDraw.Draw(card).text((18, 85), "CPU INTEGRATION TEST\nNOT EXPERIMENT APPARATUS", fill="white")
    card.save(image_path)
    scene_images[scene_id] = {"path": str(image_path), "provenance": "synthetic CPU test card", "review_status": "integration_fixture"}
pipeline = VideoPipeline(generator="dummy", output_dir=RUN_DIR / "video")
preflight = pipeline.preflight(plan_file, scene_images=scene_images)
print("Required images resolved:", list(preflight["images"]))

## Generate, inspect, and play
Every output clip is decoded; frame counts and continuity are checked. The manifest records source IDs, prompts, image hashes, generator settings, and review status.

In [ ]:
result = pipeline.run(plan_file, scene_images=scene_images)
assert result.manifest["experiment_id"] == EXPERIMENT_ID
assert result.manifest["output_kind"] == "integration_output"
assert result.manifest["media"]["frame_count"] == sum(c["media"]["frame_count"] for c in result.manifest["clips"])
for index, clip in enumerate(result.manifest["clips"]):
    if not plan["clips"][index]["new_scene"]:
        assert clip["input_image"]["sha256"] == result.manifest["clips"][index-1]["last_frame_sha256"]
print(result.video_path)
print(result.manifest_path)
display(Video(str(result.video_path), embed=True))

## Verify rejection and safe resume

In [ ]:
blocked = catalog.plan("chem_9_001")
assert blocked["status"] == "blocked" and blocked["clips"] == []
try:
    pipeline.run(blocked)
except Exception as exc:
    print("Blocked plan rejected before generation:", type(exc).__name__)
else:
    raise AssertionError("Blocked plan was accepted")
resumed = pipeline.run(plan_file, scene_images=scene_images)
assert all(c["reused"] for c in resumed.manifest["clips"])
print("CPU workflow passed; visual/scientific video assessment remains pending.")